**`ingest_buildings`**

Script examples to import building datasets

# Configure

In [ ]:
import argparse

from openplaces.io.ingester import Ingester
from openplaces.recipe import get_recipe_by_id
from openplaces.utils import pretty_print

In [ ]:
# Define arguments
parser = argparse.ArgumentParser(description='Ingest buildings using a recipe')
parser.add_argument(
    '--recipe_id',
    help='Identifier of the recipe (e.g., "US_building-nsi-2022")',
)
parser.add_argument(
    '--admin_ids',
    help='Administrative unit IDs to ingest (e.g., "US-RI")',
    nargs='*',
)
parser.add_argument(
    '--reprocess',
    help='Reprocess input data from downloaded file',
    action='store_true',
)
parser.add_argument(
    '--redownload',
    help='Redownload input data from original source',
    action='store_true',
)
parser.add_argument(
    '--verbose',
    help='If True, print outputs while processing data',
    action='store_true',
)
parser.add_argument(
    '--keep_unzipped',
    help='If True, keeps unzipped datasets in heap folder after processing',
    action='store_true',
)

# Test arguments

In [ ]:
ARGS_TEST = (
    # Global building recipe #1: OpenBuildingsMap
    # '--recipe_id building-obm-2025 '
    # Global building recipe #2: Global Human Settlement Open Buildings
    # '--recipe_id building-ghs-v1 '
    # Location #1: Brunswick, NC (flood/hurricane risk case)
    # '--admin_ids US-NC-BS '
    # Location #1: Gibratar (small, urban)
    # '--admin_ids GI '
    # Building recipe #1: US National Structure Inventory
    '--recipe_id US_building-nsi-2022 '
    # Building recipe #2: FEMA USA Structures
    # '--recipe_id US_building-fema-2023 '
    # Building recipe #3: Microsoft US building footprints
    # '--recipe_id US_building-microsoft-v2 '
    # Building recipe #4: North Carolina, US, building footprints
    # '--recipe_id US-NC_building-ncdps-2023 '
    # Location #1: Newport county, RI (fastest US county)
    '--admin_ids US-RI-NE '
    # Location #2: Brunswick and Buncombe, NC (flood/hurricane risk cases)
    # '--admin_ids US-NC-BS US-NC-BO '
    # Location #3: Harris and Jefferson, TX (hurricane risk cases)
    # '--admin_ids US-TX-RR US-TX-JE '
    # Location #4: Oneida and Vilas counties, WI (lake hedonics case study)
    # '--admin_ids US-WI-ON US-WI-VI '
    # Location #5: Hillsborough, Pinellas, and Manatee counties, FL (Tampa bay)
    # '--admin_ids US-FL-HL US-FL-MN US-FL-PI '
    # Location #6: Eastern North Carolina (CHEER building inventory)
    # '--admin_ids US-NC-BA US-NC-BT US-NC-BL US-NC-BS US-NC-CD US-NC-CE US-NC-CW US-NC-CM US-NC-CN US-NC-CU US-NC-CI US-NC-DE US-NC-DP US-NC-ED US-NC-FR US-NC-GT US-NC-GE US-NC-HL US-NC-HT US-NC-HD US-NC-HO US-NC-HE US-NC-JH US-NC-JN US-NC-LN US-NC-MR US-NC-NA US-NC-NE US-NC-NO US-NC-ON US-NC-PM US-NC-PU US-NC-PD US-NC-PQ US-NC-PI US-NC-RB US-NC-SP US-NC-SC US-NC-TY US-NC-WK US-NC-WR US-NC-WI US-NC-WY US-NC-WO '
    # Location #7: Coastal Texas (CHEER building inventory)
    # '--admin_ids US-TX-AA US-TX-AU US-TX-BE US-TX-BI US-TX-BK US-TX-CU US-TX-CA US-TX-CH US-TX-CO US-TX-DE US-TX-DU US-TX-FT US-TX-FR US-TX-GV US-TX-GI US-TX-HN US-TX-RR US-TX-HG US-TX-JA US-TX-JR US-TX-JE US-TX-JG US-TX-JW US-TX-KE US-TX-KL US-TX-LA US-TX-LT US-TX-LK US-TX-MD US-TX-NE US-TX-NU US-TX-OR US-TX-RF US-TX-SP US-TX-SR US-TX-TY US-TX-VI US-TX-WR US-TX-WH US-TX-WB US-TX-WN US-TX-WY '
    # Processing flags
    '--reprocess '
    # '--redownload '
    '--verbose '
    '--keep_unzipped '
)

# Convert argument string to list of strings
args_list = [x for x in ARGS_TEST.split(' ') if x != '']

# Parse list of arguments
args = parser.parse_args(args_list)

# Display arguments to check if parsing worked as expected
args

In [ ]:
# Show recipe parameters
pretty_print(get_recipe_by_id(args.recipe_id))

# Ingest building data

In [ ]:
ingester = Ingester(args.recipe_id, args.admin_ids, verbose=args.verbose)

In [ ]:
ingester.ingest(
    reprocess=args.reprocess,
    redownload=args.redownload,
    keep_unzipped=args.keep_unzipped,
)

---
# Convert to script

*The above line and heading identify the end of the script.*

*Code below this marker will not be included in the converted `.py` script.*

In [ ]:
from openplaces.flow import convert_to_script, test_script

COMMIT = True
# If True, writes `.py` scripts to 'scripts/.../'.
# If False, writes a test version of the script to 'scripts/_test/...'

In [ ]:
convert_to_script(commit=COMMIT)

# Test script

In [ ]:
test_script(*args_list, committed=COMMIT)

# Inspect results

## Show full map

In [ ]:
import contextily as cx
import matplotlib.pyplot as plt
import shapely

from openplaces.api import get_admin, read_entities
from openplaces.core.schema import AdminId
from openplaces.io.admin import find_admin_recipe_id

if ingester.admin_ids_to_save:
    last_saved_admin_id = ingester.admin_ids_to_save[-1]
    print(last_saved_admin_id)

    level = AdminId(last_saved_admin_id).get_level()
    # Find the key in an official admin dataset
    admin_recipe_id = None
    if admin_recipe_id is None and level > 1:
        admin_recipe_id = find_admin_recipe_id(ingester.recipe['admin_id'], level)
    admin = get_admin(
        last_saved_admin_id, level=level, recipe=admin_recipe_id, geom=True
    )

    buildings = read_entities(ingester.recipe, last_saved_admin_id, geom=True)

    fig, ax = plt.subplots(figsize=(10, 10))

    if isinstance(buildings.geometry.iloc[0], shapely.geometry.Point):
        buildings.plot(ax=ax, facecolor='red', markersize=2, linewidth=0, alpha=0.7)
    elif isinstance(
        buildings.geometry.iloc[0],
        (shapely.geometry.Polygon, shapely.geometry.MultiPolygon),
    ):
        if len(buildings) > 250000:
            print('>250K building polygons to plot. Taking sample')
            _buildings_plot = buildings.sample(250000)
        else:
            _buildings_plot = buildings
        buildings.plot(
            ax=ax, color='skyblue', edgecolor='blue', linewidth=0.5, alpha=0.5
        )

    admin.boundary.plot(ax=ax, color='black', linewidth=0.25)
    ax.set_title(
        f'{len(buildings):,d} buildings in '
        + admin.loc[last_saved_admin_id, 'name']
        + f' ({last_saved_admin_id}) in recipe: {args.recipe_id}'
    )
    ax.axis('off')
    cx.add_basemap(
        ax,
        crs=buildings.crs,
        source=cx.providers.Esri.WorldImagery,
        alpha=0.5,
    )

## Show random building geometry with attributes

In [ ]:
from openplaces.viz import show_geometry_context

if ingester.admin_ids_to_save:
    show_geometry_context(buildings, buildings.sample().index[0])